# 03 · Ciclo de vida de una canción

Comparamos la **forma de onda** de un hit **viral** (entra fuerte, pico rápido, caída)
contra la de un tema de **catálogo** (se sostiene mucho tiempo, sin pico marcado).

Las series se arman en memoria desde `data/charts_argentina.csv` con
`construir_serie()` + `interpolar_serie()` (no se persisten como CSV).

## 0. Armado de las series

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import (
    construir_serie, interpolar_serie, graficar_serie, resumen_estadistico,
    calcular_fft, graficar_espectro, descomponer_MA7_MA21, energia_serie,
    autocorrelacion, test_estacionariedad,
)

# --- Parámetros (cambiar acá las canciones si hace falta) ---
CANCION_VIRAL    = "C.R.O: Bzrp Music Sessions, Vol. 29"
CANCION_CATALOGO = "Pasos Al Costado"
REGION           = "Argentina"
CHART            = "top200"
# ------------------------------------------------------------

df = pd.read_csv("../data/charts_argentina.csv", parse_dates=["date"])

def armar_serie(cancion):
    cruda = construir_serie(df, {"title": cancion, "region": REGION, "chart": CHART,
                                 "valor": "streams", "agregado": "sum"})
    serie = interpolar_serie(cruda)
    huecos = len(serie) - cruda.index.nunique()
    print(f"{cancion}: {len(serie)} días | {huecos} huecos interpolados | "
          f"{serie.index.min().date()} → {serie.index.max().date()}")
    return serie

serie_viral    = armar_serie(CANCION_VIRAL)
serie_catalogo = armar_serie(CANCION_CATALOGO)

## 1. Exploratorio

Forma de onda y estadísticos descriptivos de cada serie.

In [ ]:
graficar_serie(serie_viral, f"Streams diarios — {CANCION_VIRAL} (viral)")
plt.show()
graficar_serie(serie_catalogo, f"Streams diarios — {CANCION_CATALOGO} (catálogo)")
plt.show()

In [ ]:
print("VIRAL   :", {k: round(v) for k, v in resumen_estadistico(serie_viral).items()})
print("CATÁLOGO:", {k: round(v) for k, v in resumen_estadistico(serie_catalogo).items()})

## 2. Fourier

FFT de la serie viral y su espectro de magnitud.

In [ ]:
frecs, mags = calcular_fft(serie_viral)
graficar_espectro(frecs, mags, f"Espectro — {CANCION_VIRAL}")
plt.axvline(1/7, color="red", ls="--", lw=1, label="1/7 (semanal)")
plt.legend(); plt.show()

f_pico = frecs[np.argmax(mags)]
print(f"Frecuencia dominante: {f_pico:.4f} ciclos/día → período {1/f_pico:.1f} días")

**Qué frecuencias dominan.** La energía del espectro se concentra en las
**frecuencias muy bajas** (períodos largos): es la firma del transitorio del ciclo de
vida —la subida y la caída lenta del pico— que domina la señal. Sobre ese fondo
aparece un componente en torno a **1/7 ciclos/día** (período ≈ 7), la
**estacionalidad semanal** del consumo (más streams los fines de semana). No hay
picos de alta frecuencia relevantes: la serie es suave, sin oscilaciones rápidas.

## 3. Filtrado (MA7 / MA21)

Descomponemos la serie viral en tendencia (MA21), componente semanal (MA7 − MA21) y
residuo (serie − MA7).

In [ ]:
desc = descomponer_MA7_MA21(serie_viral)

fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
ax[0].plot(serie_viral.index, serie_viral.values); ax[0].set_title("Serie original (viral)")
ax[1].plot(desc["tendencia"]); ax[1].set_title("Tendencia (MA21)")
ax[2].plot(desc["estacional"]); ax[2].set_title("Estacional (MA7 − MA21)")
ax[3].plot(desc["residuo"]); ax[3].set_title("Residuo (serie − MA7)")
for a in ax: a.set_ylabel("streams")
plt.tight_layout(); plt.show()

## 4. Energía y Parseval

Energía en tiempo vs. frecuencia (chequeo de Parseval) **antes** y **después** del
filtrado. La señal filtrada es la suavizada `MA7 = tendencia + estacional` (le sacamos
el residuo de alta frecuencia).

In [ ]:
filtrada = desc["tendencia"] + desc["estacional"]   # = MA7 (sin el residuo)

e_antes = energia_serie(serie_viral)
e_desp  = energia_serie(filtrada)

print("ANTES del filtrado:")
print(f"    E_tiempo = {e_antes['energia_tiempo']:.4e} | E_frec = {e_antes['energia_frecuencia']:.4e} "
      f"| Parseval OK: {e_antes['parseval_ok']}")
print("DESPUÉS del filtrado (MA7):")
print(f"    E_tiempo = {e_desp['energia_tiempo']:.4e} | E_frec = {e_desp['energia_frecuencia']:.4e} "
      f"| Parseval OK: {e_desp['parseval_ok']}")
print(f"\nEnergía retenida tras filtrar: {100 * e_desp['energia_tiempo'] / e_antes['energia_tiempo']:.2f}% "
      f"(el resto es el residuo de alta frecuencia)")

## 5. Autocorrelación (extra de esta pregunta)

Comparamos la ACF de la serie viral con la de catálogo: la forma de la
autocorrelación resume qué tan "predecible" y estructurada es cada señal.

In [ ]:
NLAGS = 30
ac_viral = autocorrelacion(serie_viral, NLAGS)
ac_cat   = autocorrelacion(serie_catalogo, NLAGS)

fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
ax[0].stem(range(len(ac_viral)), ac_viral); ax[0].set_title("ACF — viral")
ax[1].stem(range(len(ac_cat)), ac_cat);     ax[1].set_title("ACF — catálogo")
for a in ax:
    a.axvline(7, color="red", ls="--", lw=1); a.set_xlabel("lag (días)")
ax[0].set_ylabel("autocorrelación")
plt.tight_layout(); plt.show()

print(f"ACF lag 1  — viral: {ac_viral[1]:.3f} | catálogo: {ac_cat[1]:.3f}")
print(f"ACF lag 7  — viral: {ac_viral[7]:.3f} | catálogo: {ac_cat[7]:.3f}")
print(f"ACF lag 30 — viral: {ac_viral[30]:.3f} | catálogo: {ac_cat[30]:.3f}")

**Cómo difiere el patrón.** En la **viral**, la ACF arranca muy alta y **decae
lento y monótono** con el lag: cada día se parece mucho al anterior porque la serie
está dominada por la tendencia suave del pico (subida/caída), no por oscilaciones.
En la de **catálogo**, la autocorrelación también es alta pero se mantiene más plana
en el largo plazo (meseta persistente durante años) y deja ver mejor la **modulación
semanal** (rebote alrededor del lag 7), al no estar tapada por un transitorio
dominante. En síntesis: la viral está gobernada por un evento (baja frecuencia), la de
catálogo por un régimen estacionario con estructura semanal.

## 6. Estacionariedad y procesamiento tiempo-frecuencia

Se evaluó la estacionariedad de la serie viral con `test_estacionariedad()` (ADF +
KPSS). El resultado confirma que **la serie NO es estacionaria**: sus estadísticos
(media, varianza) cambian fuertemente a lo largo del tiempo por culpa del pico —hay un
régimen de "explosión" durante el ascenso y otro muy distinto en la cola de la caída.

Esto conecta con el **procesamiento tiempo-frecuencia** del apunte: como el contenido
espectral de la señal **cambia con el tiempo** (durante el pico hay un transitorio de
banda ancha que no está presente en la meseta posterior), un análisis de Fourier
"global" es insuficiente por sí solo —promedia comportamientos de distintas épocas— y
por eso lo complementamos con **filtrado local en el tiempo** (MA7/MA21), que separa
tendencia y estacionalidad respetando *cuándo* ocurre cada cosa.

In [ ]:
est = test_estacionariedad(serie_viral)
print("ADF :", est["adf"])
print("KPSS:", est["kpss"])
print(f"\n→ ¿Estacionaria? ADF dice {est['adf']['estacionaria']}, "
      f"KPSS dice {est['kpss']['estacionaria']}")

> **Nota (wavelets).** Se consideró aplicar *wavelets* (algoritmo de Mallat) para
> separar el pico del resto sin perder la localización temporal, pero se descartó como
> sección aparte porque el filtrado con MA7/MA21 ya resuelve esa separación para el
> alcance de este trabajo.

## 7. Métricas del ciclo de vida (cálculo manual)

Día del pico, valor máximo, duración, pendientes de subida/caída y área bajo la curva,
calculados a mano sobre la serie viral.

In [ ]:
s = serie_viral
_trapz = getattr(np, "trapezoid", None) or np.trapz

idx_pico     = s.values.argmax()
dia_pico     = s.index[idx_pico]
valor_max    = s.values[idx_pico]
duracion     = len(s)

dia_inicio, val_inicio = s.index[0], s.values[0]
dia_fin, val_fin       = s.index[-1], s.values[-1]
dias_subida  = max((dia_pico - dia_inicio).days, 1)
dias_caida   = max((dia_fin - dia_pico).days, 1)

pend_subida  = (valor_max - val_inicio) / dias_subida
pend_caida   = (val_fin - valor_max) / dias_caida

dias_rel     = (s.index - dia_inicio).days.to_numpy()
area         = _trapz(s.values, x=dias_rel)

print(f"Día del pico:        {dia_pico.date()}")
print(f"Valor máximo:        {valor_max:,.0f} streams")
print(f"Duración en chart:   {duracion} días")
print(f"Pendiente subida:    {pend_subida:,.0f} streams/día (en {dias_subida} días)")
print(f"Pendiente caída:     {pend_caida:,.0f} streams/día (en {dias_caida} días)")
print(f"|subida| / |caída|:  {abs(pend_subida) / abs(pend_caida):.1f}×")
print(f"Área bajo la curva:  {area:,.0f} streams·día")

## 8. Conclusión: viral vs. catálogo

La diferencia está en la **forma de la curva**:

- **Viral** — señal claramente **no estacionaria** dominada por un **transitorio**:
  ascenso muy pronunciado (pendiente de subida varias veces mayor, en valor absoluto,
  que la de caída), un **pico** marcado y una caída sostenida durante meses hasta salir
  del chart. En frecuencia se traduce en energía concentrada en baja frecuencia; en la
  ACF, en un decaimiento lento y monótono. Es la firma de un tema que "explota" con el
  lanzamiento y después se apaga.

- **Catálogo** — se comporta casi como una **meseta estacionaria** de bajo nivel: entra
  sin salto brusco, se sostiene en una banda angosta de streams durante años, sin un
  pico dominante (pendientes de subida y caída chicas), y deja ver mejor la modulación
  **semanal**. Aunque su pico nunca se acerque al de la viral, su **área bajo la curva**
  a largo plazo puede ser mayor: acumula streams moderados durante mucho más tiempo.

En términos de la materia: la viral es un **evento** (transitorio, tiempo-frecuencia
cambiante) y la de catálogo es un **régimen** (cuasi-estacionario, con estructura
periódica semanal).